# Customer Churn Prediction — Part 2: Preprocessing & Feature Engineering
**Objective:** Build clean, leakage-free data transformations and domain-specific feature engineering.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('../data/dataset.csv')
# Drop identifier
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Handle TotalCharges empty space
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)

# Separate Target
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# PREVENT DATA LEAKAGE: Split first!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

### Domain Feature Engineering
1. `tenure_group`: Group tenure into 5 lifecycle bins.
2. `total_services`: Sum of active value-add tech/security/streaming products.
3. `charges_per_tenure`: Progression of spending rate per month.
4. `streaming_bundle`: Heavy entertainment consumer flag.

In [2]:
def engineer_features(data):
    df_fe = data.copy()
    # 1. Tenure bins
    bins = [-1, 12, 24, 48, 60, 100]
    labels = ['0-12m', '13-24m', '25-48m', '49-60m', '>60m']
    df_fe['tenure_group'] = pd.cut(df_fe['tenure'], bins=bins, labels=labels).astype(str)
    
    # 2. Total service count
    services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df_fe['total_services'] = df_fe[services].apply(lambda row: sum(1 for v in row if str(v).lower() == 'yes'), axis=1)
    
    # 3. Charges per tenure
    tenure_safe = df_fe['tenure'].replace(0, 1)
    df_fe['charges_per_tenure'] = df_fe['MonthlyCharges'] / tenure_safe
    
    return df_fe

X_train_fe = engineer_features(X_train)
print("Engineered Feature Columns:", X_train_fe.columns.tolist()[-3:])